# 03_chunking_v2_recursive.ipynb

**Actividad de Grado MIA UC**  
**Sistema Inteligente de Vigilancia de Riesgos Documentales basado en RAG y LLMs**

Este notebook mejora el chunking baseline mediante una estrategia **recursive character text splitting**, que intenta cortar el texto respetando separadores naturales como párrafos, saltos de línea, oraciones y espacios.

## Objetivo
Generar una versión mejorada de chunks para RAG, manteniendo trazabilidad por documento, página y estrategia experimental.

## Salidas
- `chunks_recursive.parquet`
- `chunks_recursive.csv`
- `resumen_chunking_recursive.csv`
- `errores_chunking_recursive.csv` si aplica


In [ ]:
# =====================================================
# 03_chunking_v2_recursive.ipynb
# Actividad de Grado MIA UC
# Patricia Patiño
# =====================================================

!pip -q install pandas pymupdf python-docx pyarrow openpyxl langchain-text-splitters

## 1. Importar librerías

In [ ]:
import re
import fitz
import pandas as pd
from pathlib import Path
from docx import Document
from google.colab import files
from langchain_text_splitters import RecursiveCharacterTextSplitter

pd.set_option('display.max_colwidth', 140)

print('Librerías cargadas correctamente')

## 2. Cargar inventario y documentos

Sube:
1. `inventario_corpus.csv`
2. Los 36 documentos PDF/DOCX del corpus


In [ ]:
print('Sube inventario_corpus.csv y todos los documentos PDF/DOCX del corpus')
uploaded = files.upload()
print('\nArchivos cargados:', len(uploaded))

## 3. Leer y validar inventario

In [ ]:
inventario_path = Path('inventario_corpus.csv')

if not inventario_path.exists():
    raise FileNotFoundError('No se encontró inventario_corpus.csv. Debes subirlo junto con los documentos.')

inventario = pd.read_csv(inventario_path)

columnas_requeridas = {'doc_id', 'filename', 'extension', 'tipo_documento'}
faltantes = columnas_requeridas - set(inventario.columns)

if faltantes:
    raise ValueError(f'Faltan columnas en el inventario: {faltantes}')

display(inventario.head())
print('Total documentos en inventario:', len(inventario))
display(inventario['tipo_documento'].value_counts().reset_index())

## 4. Funciones de extracción y limpieza

Se mantiene la página en PDF para trazabilidad. En DOCX se asigna página 1 porque el formato no conserva paginación fija.

In [ ]:
def extraer_texto_pdf(path):
    paginas = []
    doc = fitz.open(path)
    for page_num, page in enumerate(doc, start=1):
        texto = page.get_text('text')
        paginas.append({'page': page_num, 'text': texto})
    doc.close()
    return paginas


def extraer_texto_docx(path):
    documento = Document(path)
    parrafos = [p.text for p in documento.paragraphs if p.text.strip()]
    texto = '\n'.join(parrafos)
    return [{'page': 1, 'text': texto}]


def limpiar_texto(texto):
    if texto is None:
        return ''
    texto = texto.replace('\x00', ' ')
    texto = re.sub(r'[ \t]+', ' ', texto)
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    texto = texto.strip()
    return texto


def normalizar_nombre_archivo(nombre):
    return str(nombre).strip()

## 5. Configuración del chunking recursivo

Esta estrategia intenta cortar en este orden:
1. Doble salto de línea
2. Salto de línea
3. Punto seguido
4. Espacio
5. Carácter

Esto suele conservar mejor la coherencia semántica que cortar estrictamente cada N palabras.

In [ ]:
# Parámetros propuestos para documentos de interventoría técnica
CHUNK_SIZE_CHARS = 2200
CHUNK_OVERLAP_CHARS = 300
MIN_WORDS = 40

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_CHARS,
    chunk_overlap=CHUNK_OVERLAP_CHARS,
    separators=['\n\n', '\n', '. ', '; ', ', ', ' ', ''],
    length_function=len
)

print('Configuración chunking recursivo')
print('chunk_size_chars:', CHUNK_SIZE_CHARS)
print('chunk_overlap_chars:', CHUNK_OVERLAP_CHARS)
print('min_words:', MIN_WORDS)

## 6. Procesar documentos

In [ ]:
registros_chunks = []
errores = []

for _, row in inventario.iterrows():
    doc_id = row['doc_id']
    filename = normalizar_nombre_archivo(row['filename'])
    extension = str(row['extension']).lower().strip()
    tipo_documento = row['tipo_documento']

    path = Path(filename)

    if not path.exists():
        errores.append({
            'doc_id': doc_id,
            'filename': filename,
            'error': 'Archivo no encontrado en la sesión de Colab'
        })
        continue

    try:
        if extension == '.pdf':
            paginas = extraer_texto_pdf(path)
        elif extension == '.docx':
            paginas = extraer_texto_docx(path)
        else:
            errores.append({
                'doc_id': doc_id,
                'filename': filename,
                'error': f'Extensión no soportada: {extension}'
            })
            continue

        chunk_counter = 1

        for pagina in paginas:
            page = pagina['page']
            texto_limpio = limpiar_texto(pagina['text'])

            if len(texto_limpio.split()) < MIN_WORDS:
                continue

            chunks = splitter.split_text(texto_limpio)

            for chunk in chunks:
                chunk = limpiar_texto(chunk)
                palabras = len(chunk.split())

                if palabras < MIN_WORDS:
                    continue

                registros_chunks.append({
                    'doc_id': doc_id,
                    'filename': filename,
                    'tipo_documento': tipo_documento,
                    'page': page,
                    'chunk_id': f'{doc_id}_RCHUNK_{chunk_counter:03d}',
                    'chunk_text': chunk,
                    'chunk_size_words': palabras,
                    'chunk_size_chars': len(chunk),
                    'chunking_strategy': 'recursive_character_splitter',
                    'chunk_size_param_chars': CHUNK_SIZE_CHARS,
                    'overlap_param_chars': CHUNK_OVERLAP_CHARS
                })

                chunk_counter += 1

    except Exception as e:
        errores.append({
            'doc_id': doc_id,
            'filename': filename,
            'error': str(e)
        })

chunks_recursive_df = pd.DataFrame(registros_chunks)
errores_recursive_df = pd.DataFrame(errores)

print('Total chunks recursivos generados:', len(chunks_recursive_df))
print('Documentos con error:', len(errores_recursive_df))

display(chunks_recursive_df.head())

if len(errores_recursive_df) > 0:
    display(errores_recursive_df)

## 7. Validaciones de calidad

In [ ]:
if chunks_recursive_df.empty:
    raise ValueError('No se generaron chunks. Revisa archivos cargados, extensiones o inventario.')

resumen_recursive = (
    chunks_recursive_df.groupby(['doc_id', 'filename', 'tipo_documento'])
    .agg(
        cantidad_chunks=('chunk_id', 'count'),
        total_palabras=('chunk_size_words', 'sum'),
        promedio_palabras_chunk=('chunk_size_words', 'mean'),
        min_palabras=('chunk_size_words', 'min'),
        max_palabras=('chunk_size_words', 'max')
    )
    .reset_index()
)

print('Documentos procesados:', resumen_recursive['doc_id'].nunique())
print('Chunks totales:', len(chunks_recursive_df))
print('Promedio palabras por chunk:', round(chunks_recursive_df['chunk_size_words'].mean(), 2))

display(resumen_recursive.head())
display(chunks_recursive_df['chunk_size_words'].describe())

## 8. Estadísticas por tipo documental

In [ ]:
stats_tipo_recursive = (
    chunks_recursive_df.groupby('tipo_documento')
    .agg(
        documentos=('doc_id', 'nunique'),
        chunks=('chunk_id', 'count'),
        palabras_totales=('chunk_size_words', 'sum'),
        promedio_palabras_chunk=('chunk_size_words', 'mean')
    )
    .reset_index()
)

display(stats_tipo_recursive)

## 9. Inspección manual

Revisa algunos chunks para validar si conservan coherencia semántica.

In [ ]:
muestra = chunks_recursive_df.sample(min(5, len(chunks_recursive_df)), random_state=42)

for _, row in muestra.iterrows():
    print('='*100)
    print('chunk_id:', row['chunk_id'])
    print('documento:', row['filename'])
    print('tipo:', row['tipo_documento'])
    print('página:', row['page'])
    print('palabras:', row['chunk_size_words'])
    print('-'*100)
    print(row['chunk_text'][:1200])
    print()

## 10. Comparación opcional contra baseline

Si tienes el archivo `resumen_chunking.csv` del notebook anterior, súbelo en la misma sesión y ejecuta esta celda. Si no lo tienes, puedes omitirla.

In [ ]:
baseline_path = Path('resumen_chunking.csv')

if baseline_path.exists():
    resumen_baseline = pd.read_csv(baseline_path)
    comparacion = pd.DataFrame({
        'estrategia': ['fixed_words_overlap', 'recursive_character_splitter'],
        'documentos_procesados': [resumen_baseline['doc_id'].nunique(), resumen_recursive['doc_id'].nunique()],
        'chunks_totales': [resumen_baseline['cantidad_chunks'].sum(), resumen_recursive['cantidad_chunks'].sum()],
        'promedio_chunks_documento': [resumen_baseline['cantidad_chunks'].mean(), resumen_recursive['cantidad_chunks'].mean()],
        'promedio_palabras_chunk': [resumen_baseline['promedio_palabras_chunk'].mean(), resumen_recursive['promedio_palabras_chunk'].mean()]
    })
    display(comparacion)
else:
    print('No se encontró resumen_chunking.csv. Comparación omitida.')

## 11. Guardar artefactos

Estos archivos serán usados en el Notebook 04 de embeddings.

In [ ]:
chunks_recursive_df.to_parquet('chunks_recursive.parquet', index=False)
chunks_recursive_df.to_csv('chunks_recursive.csv', index=False, encoding='utf-8-sig')
resumen_recursive.to_csv('resumen_chunking_recursive.csv', index=False, encoding='utf-8-sig')

if len(errores_recursive_df) > 0:
    errores_recursive_df.to_csv('errores_chunking_recursive.csv', index=False, encoding='utf-8-sig')

print('Archivos generados:')
print('- chunks_recursive.parquet')
print('- chunks_recursive.csv')
print('- resumen_chunking_recursive.csv')
if len(errores_recursive_df) > 0:
    print('- errores_chunking_recursive.csv')

## 12. Descargar artefactos

In [ ]:
files.download('chunks_recursive.parquet')
files.download('chunks_recursive.csv')
files.download('resumen_chunking_recursive.csv')

if len(errores_recursive_df) > 0:
    files.download('errores_chunking_recursive.csv')